# AllLegs — Parallel PPO (multi-environment experience collection)

## Where is the bottleneck?

In `AllLegs_def.ipynb` training is sequential:

```
for each episode:
    rollout_train()   ← 800 steps × (MuJoCo 5×step ≈ 2.5 ms + NN ≈ 0.1 ms)
                      ≈ 2.1 s / episode   ← BOTTLENECK
    if buf full:
        update_ppo()  ← fast, GPU/CPU, ≈ 0.5 s for 10 epochs
```

MuJoCo is single-threaded C — it dominates wall-clock time.  
The neural network is tiny (24→64→8) and negligible.

## Solution: N parallel MuJoCo workers

```
main process (CPU)                subprocess × N_WORKERS
──────────────────                ─────────────────────────────────────────
broadcast weights ──────────────► worker 0: own MjModel → rollout → results
                  ──────────────► worker 1: own MjModel → rollout → results
                  ──────────────► worker 2: own MjModel → rollout → results
                  ──────────────► worker 3: own MjModel → rollout → results
                  ◄──────────────  all results arrive in parallel
concatenate into PPOBuffer
PPO update (when buffer full)
```

**Expected speedup:** ~N_WORKERS × (limited by memory bandwidth + process overhead)

**macOS note:** MuJoCo is not fork-safe. We use `spawn` context explicitly.

In [1]:
# ── Write the worker module to disk ──────────────────────────────────────────
# The worker runs in a SUBPROCESS, so all its code must live in an importable
# Python file — Jupyter cell definitions are not visible to child processes.

_WORKER_CODE = '''
"""
_parallel_worker.py
Subprocess worker for parallel PPO experience collection.
Receives current policy weights + config, runs one episode, returns transitions.
"""
import os, math
import numpy as np
import torch
import torch.nn as nn
import mujoco


def _worker(worker_id: int, weights_numpy: dict, cfg: dict) -> dict:
    """
    Collect one episode of experience in an independent MuJoCo instance.

    Parameters
    ----------
    worker_id     : int  — used for seeding / debugging
    weights_numpy : dict[str, ndarray] — current policy weights as numpy arrays
    cfg           : dict — all hyperparameters (see build_cfg() in the notebook)

    Returns
    -------
    dict with lists: states, actions, log_probs, rewards, values, dones
         and scalars: last_value, dist, tot_r
    """
    # ── Unpack config ────────────────────────────────────────────────────────
    STATE_DIM    = cfg["STATE_DIM"]
    ACTION_DIM   = cfg["ACTION_DIM"]
    HIDDEN_DIM   = cfg["HIDDEN_DIM"]
    LOG_STD_INIT = cfg["LOG_STD_INIT"]
    LOG_STD_MIN  = cfg["LOG_STD_MIN"]
    LOG_STD_MAX  = cfg["LOG_STD_MAX"]

    SUBSTEPS     = cfg["SUBSTEPS"]
    SETTLE_TIME  = cfg["SETTLE_TIME"]
    EPISODE_DUR  = cfg["EPISODE_DUR"]
    XML_PATH     = cfg["XML_PATH"]

    HIP_INIT     = cfg["HIP_INIT"]
    KNEE_INIT    = cfg["KNEE_INIT"]
    DELTA_HIP    = cfg["DELTA_HIP"]
    DELTA_KNEE   = cfg["DELTA_KNEE"]
    MAX_DDELTA   = cfg["MAX_DDELTA"]
    DELTA_LIMIT  = cfg["DELTA_LIMIT"]

    TARGET_VX    = cfg["TARGET_VX"]
    VX_TRACK_W   = cfg["VX_TRACK_W"]
    VX_TOL       = cfg["VX_TOL"]
    ALIVE_BONUS  = cfg["ALIVE_BONUS"]
    JVEL_W       = cfg["JVEL_W"]
    JEXC_W       = cfg["JEXC_W"]
    ACTION_REG_W = cfg["ACTION_REG_W"]
    DELTA_REG_W  = cfg["DELTA_REG_W"]
    ALIGN_W      = cfg["ALIGN_W"]
    YAW_W        = cfg["YAW_W"]

    OFFSETS  = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
    JOINT_LO = np.array([HIP_INIT - DELTA_HIP,  KNEE_INIT - DELTA_KNEE] * 4, dtype=np.float32)
    JOINT_HI = np.array([HIP_INIT + DELTA_HIP,  KNEE_INIT + DELTA_KNEE] * 4, dtype=np.float32)

    # ── Each subprocess gets its own MuJoCo instance ─────────────────────────
    # MuJoCo model/data are not serialisable — must be created fresh per process.
    mjmodel = mujoco.MjModel.from_xml_path(XML_PATH)
    mjdata  = mujoco.MjData(mjmodel)
    CTRL_DT = mjmodel.opt.timestep * SUBSTEPS

    # ── Reconstruct ActorCritic from transmitted weights ──────────────────────
    # We define the class inline so the worker file is fully self-contained.
    class ActorCritic(nn.Module):
        def __init__(self):
            super().__init__()
            self.pi = nn.Sequential(
                nn.Linear(STATE_DIM, HIDDEN_DIM), nn.ReLU(),
                nn.Linear(HIDDEN_DIM, ACTION_DIM),
            )
            self.log_std = nn.Parameter(torch.full((ACTION_DIM,), LOG_STD_INIT))
            self.vf = nn.Sequential(
                nn.Linear(STATE_DIM, 256), nn.ReLU(),
                nn.Linear(256, 256),       nn.ReLU(),
                nn.Linear(256, 1),
            )

        def get_action(self, s):
            mean    = self.pi(s)
            log_std = self.log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
            std     = log_std.exp()
            raw     = mean + std * torch.randn_like(std)
            action  = torch.tanh(raw)
            log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                        - log_std - 0.5 * math.log(2 * math.pi))
            log_prob -= torch.log(1 - action.pow(2) + 1e-6)
            log_prob  = log_prob.sum(-1)
            value = self.vf(s).squeeze(-1)
            return action, log_prob, value

    ac = ActorCritic()
    # Convert numpy arrays back to tensors and load
    state_dict = {k: torch.from_numpy(v) for k, v in weights_numpy.items()}
    ac.load_state_dict(state_dict)
    ac.eval()

    # ── Simulation helpers ────────────────────────────────────────────────────
    def orientation():
        qw, qx, qy, qz = mjdata.qpos[3:7]
        roll  = math.atan2(2*(qw*qx + qy*qz), 1 - 2*(qx**2 + qy**2))
        pitch = math.asin(float(np.clip(2*(qw*qy - qz*qx), -1, 1)))
        yaw   = math.atan2(2*(qw*qz + qx*qy), 1 - 2*(qy**2 + qz**2))
        return roll, pitch, yaw

    def build_state(delta):
        return np.concatenate([
            mjdata.qpos[7:15].astype(np.float32),
            mjdata.qvel[6:14].astype(np.float32),
            delta.astype(np.float32),
        ])

    def apply_ddelta(ctrl, delta, ddelta):
        delta_new = np.clip(delta + ddelta * MAX_DDELTA, -DELTA_LIMIT, DELTA_LIMIT)
        ctrl_new  = np.clip(ctrl  + delta_new, JOINT_LO, JOINT_HI)
        return ctrl_new, delta_new

    def is_done():
        _, p, _ = orientation()
        return mjdata.qpos[2] < 0.03 or abs(p) > math.radians(45)

    def reset_sim():
        ip = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
        mujoco.mj_resetData(mjmodel, mjdata)
        mjdata.qpos[7:15] = ip
        mjdata.qpos[2]    = 0.10
        mujoco.mj_forward(mjmodel, mjdata)
        for _ in range(int(SETTLE_TIME / mjmodel.opt.timestep)):
            mjdata.ctrl[:] = ip
            mujoco.mj_step(mjmodel, mjdata)
        return ip.copy(), np.zeros(ACTION_DIM, dtype=np.float32)

    # ── Rollout ───────────────────────────────────────────────────────────────
    ctrl, delta = reset_sim()
    x0          = mjdata.qpos[0]
    _, _, yaw0  = orientation()
    tot_r       = 0.0
    s           = build_state(delta)

    states, actions, log_probs, rewards, values, dones = [], [], [], [], [], []

    done = False
    for _ in range(int(EPISODE_DUR / CTRL_DT)):
        st = torch.FloatTensor(s).unsqueeze(0)
        with torch.no_grad():
            action, log_prob, value = ac.get_action(st)
        an = action.squeeze(0).numpy()
        lp = log_prob.item()
        v  = value.item()

        ctrl, delta = apply_ddelta(ctrl, delta, an)
        mjdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(mjmodel, mjdata)

        roll, pitch, yaw = orientation()
        dyaw     = (yaw - yaw0 + math.pi) % (2 * math.pi) - math.pi
        vx       = float(mjdata.qvel[0])
        vy       = abs(float(mjdata.qvel[1]))
        jvel_mag = float(np.mean(np.abs(mjdata.qvel[6:14])))
        jpos     = mjdata.qpos[7:15].astype(np.float32)
        excursion = float(np.mean(np.abs(jpos - OFFSETS)))

        vx_err = max(0.0, abs(vx - TARGET_VX) - VX_TOL)
        r = (-VX_TRACK_W * vx_err**2
             + ALIVE_BONUS
             - 0.10 * vy
             - ACTION_REG_W * float(np.sum(an**2))
             - DELTA_REG_W  * float(np.sum(delta**2))
             + JVEL_W       * jvel_mag
             + JEXC_W       * excursion
             - ALIGN_W      * (roll**2 + pitch**2)
             - YAW_W        * dyaw**2)
        tot_r += r
        done = is_done()
        if done:
            r -= 5.0

        states.append(s); actions.append(an)
        log_probs.append(lp); rewards.append(r)
        values.append(v); dones.append(float(done))
        s = build_state(delta)
        if done:
            break

    if done:
        last_value = 0.0
    else:
        with torch.no_grad():
            last_value = ac.vf(torch.FloatTensor(s).unsqueeze(0)).squeeze().item()

    return {
        "states":     states,
        "actions":    actions,
        "log_probs":  log_probs,
        "rewards":    rewards,
        "values":     values,
        "dones":      dones,
        "last_value": last_value,
        "dist":       mjdata.qpos[0] - x0,
        "tot_r":      tot_r,
    }
'''

with open('_parallel_worker.py', 'w') as f:
    f.write(_WORKER_CODE)
print('Worker module written to _parallel_worker.py')

Worker module written to _parallel_worker.py


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import mujoco
import mediapy as media
import matplotlib.pyplot as plt
from tqdm import trange

# Use spawn context — required on macOS because MuJoCo is not fork-safe.
# (On macOS, Python 3.8+ already defaults to spawn, but we set it explicitly.)
import multiprocessing
multiprocessing.set_start_method('spawn', force=True)
from concurrent.futures import ProcessPoolExecutor

os.makedirs('movies',       exist_ok=True)
os.makedirs('models',       exist_ok=True)
os.makedirs('plots',        exist_ok=True)

# ── Hyper-parameters (same as AllLegs_def) ────────────────────────────────────
N_WORKERS    = 4             # number of parallel MuJoCo processes (~4× speedup)
N_ROUNDS     = 500          # training rounds (1 round = N_WORKERS episodes)

ALIGN_W      = 0.00
YAW_W        = 0.20
ALIVE_BONUS  = 0.05

MAX_DDELTA   = 0.02
DELTA_LIMIT  = 0.05

ACTION_REG_W = 0.0001
DELTA_REG_W  = 0.0002
JVEL_W       = 0.05
JEXC_W       = 0.6

TARGET_VX    = 1.0 / 4.0    # 0.25 m/s
VX_TRACK_W   = 4.0
VX_TOL       = 0.02

# ── PPO ───────────────────────────────────────────────────────────────────────
LR                 = 3e-4
GAMMA              = 0.99
GAE_LAMBDA         = 0.95
CLIP_EPS           = 0.2
ENTROPY_COEF       = 0.005
VF_COEF            = 0.5
MAX_GRAD           = 0.5
N_STEPS_PER_UPDATE = 4096
PPO_EPOCHS         = 10
MINIBATCH_SIZE     = 256

LOG_STD_INIT = 0.5
LOG_STD_MIN  = -2.0
LOG_STD_MAX  = 1.0

RENDER_EVERY = 50            # render every N rounds (more frequent since rounds are bigger)
LOAD_MODEL   = ''            # e.g. 'models/parallel_best.pth'

# ── Dimensions ────────────────────────────────────────────────────────────────
ACTION_DIM = 8
HIDDEN_DIM = 64
STATE_DIM  = 24

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}   |   N_WORKERS: {N_WORKERS}')

Device: cpu   |   N_WORKERS: 4


In [3]:
# ── Robot geometry + MuJoCo (main-process instance, used for eval only) ───────
HIP_INIT  =  0.90
KNEE_INIT = -1.40
DELTA_HIP  = 0.50
DELTA_KNEE = 0.30
HIP_LO,  HIP_HI  = HIP_INIT - DELTA_HIP,  HIP_INIT + DELTA_HIP
KNEE_LO, KNEE_HI = KNEE_INIT - DELTA_KNEE, KNEE_INIT + DELTA_KNEE
OFFSETS   = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
JOINT_LO  = np.array([HIP_LO, KNEE_LO] * 4, dtype=np.float32)
JOINT_HI  = np.array([HIP_HI, KNEE_HI] * 4, dtype=np.float32)

SUBSTEPS    = 5
SETTLE_TIME = 0.3
EPISODE_DUR = 8.0
XML_PATH    = os.path.abspath('dog.xml')  # absolute path so subprocesses can find it

mjmodel = mujoco.MjModel.from_xml_path(XML_PATH)
mjdata  = mujoco.MjData(mjmodel)
CTRL_DT = mjmodel.opt.timestep * SUBSTEPS
print(f'CTRL_DT = {CTRL_DT*1000:.1f} ms  |  episode = {EPISODE_DUR}s = {int(EPISODE_DUR/CTRL_DT)} steps')
print(f'N_WORKERS={N_WORKERS} → {N_WORKERS*int(EPISODE_DUR/CTRL_DT)} steps/round  '
      f'(PPO update every {N_STEPS_PER_UPDATE} steps)')

# ── Config dict passed to every worker ───────────────────────────────────────
# All hyperparameters in one serialisable dict (no tensors, no MuJoCo objects).
CFG = dict(
    STATE_DIM=STATE_DIM, ACTION_DIM=ACTION_DIM, HIDDEN_DIM=HIDDEN_DIM,
    LOG_STD_INIT=LOG_STD_INIT, LOG_STD_MIN=LOG_STD_MIN, LOG_STD_MAX=LOG_STD_MAX,
    SUBSTEPS=SUBSTEPS, SETTLE_TIME=SETTLE_TIME, EPISODE_DUR=EPISODE_DUR,
    XML_PATH=XML_PATH,
    HIP_INIT=HIP_INIT, KNEE_INIT=KNEE_INIT,
    DELTA_HIP=DELTA_HIP, DELTA_KNEE=DELTA_KNEE,
    MAX_DDELTA=MAX_DDELTA, DELTA_LIMIT=DELTA_LIMIT,
    TARGET_VX=TARGET_VX, VX_TRACK_W=VX_TRACK_W, VX_TOL=VX_TOL,
    ALIVE_BONUS=ALIVE_BONUS, JVEL_W=JVEL_W, JEXC_W=JEXC_W,
    ACTION_REG_W=ACTION_REG_W, DELTA_REG_W=DELTA_REG_W,
    ALIGN_W=ALIGN_W, YAW_W=YAW_W,
)

JOINT_NAMES = [
    "FL_hip", "FL_knee", "FR_hip", "FR_knee",
    "RL_hip", "RL_knee", "RR_hip", "RR_knee",
]
os.makedirs("trajectories", exist_ok=True)


CTRL_DT = 10.0 ms  |  episode = 8.0s = 800 steps
N_WORKERS=4 → 3200 steps/round  (PPO update every 4096 steps)


In [4]:
# ── ActorCritic (main process — used for the PPO update and eval) ─────────────
# The same architecture is reconstructed inside each worker from the weights dict.

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.pi = nn.Sequential(
            nn.Linear(STATE_DIM,  HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, ACTION_DIM),
        )
        nn.init.orthogonal_(self.pi[-1].weight, gain=0.01)
        nn.init.zeros_(self.pi[-1].bias)

        self.log_std = nn.Parameter(torch.full((ACTION_DIM,), LOG_STD_INIT))

        self.vf = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256),       nn.ReLU(),
            nn.Linear(256, 1),
        )
        nn.init.orthogonal_(self.vf[-1].weight, gain=1.0)
        nn.init.zeros_(self.vf[-1].bias)

    def forward_pi(self, s):
        mean    = self.pi(s)
        log_std = self.log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    def forward_vf(self, s):
        return self.vf(s).squeeze(-1)

    def get_action(self, s, deterministic=False):
        mean, log_std = self.forward_pi(s)
        std = log_std.exp()
        raw = mean if deterministic else mean + std * torch.randn_like(std)
        action = torch.tanh(raw)
        log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                    - log_std - 0.5 * math.log(2 * math.pi))
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        log_prob  = log_prob.sum(-1)
        value = self.forward_vf(s)
        return action, log_prob, value

    def evaluate_actions(self, s, actions_tanh):
        mean, log_std = self.forward_pi(s)
        std = log_std.exp()
        raw = torch.atanh(actions_tanh.clamp(-0.999, 0.999))
        log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                    - log_std - 0.5 * math.log(2 * math.pi))
        log_prob -= torch.log(1 - actions_tanh.pow(2) + 1e-6)
        log_prob  = log_prob.sum(-1)
        entropy = (log_std + 0.5 * math.log(2 * math.pi * math.e)).sum(-1)
        value   = self.forward_vf(s)
        return log_prob, entropy, value

    def weights_numpy(self):
        """Export state_dict as {name: ndarray} for subprocess transmission."""
        return {k: v.cpu().numpy() for k, v in self.state_dict().items()}


ac  = ActorCritic().to(device)
opt = torch.optim.Adam(ac.parameters(), lr=LR, eps=1e-5)

if LOAD_MODEL:
    ck = torch.load(LOAD_MODEL, map_location=device)
    ac.load_state_dict(ck.get('best_w') or ck['ac'])
    if 'opt' in ck: opt.load_state_dict(ck['opt'])
    best_dist = ck.get('best_dist', -np.inf)
    start_rnd = ck.get('rnd', 0) + 1
    rl, dl, cll, all_ = ck.get('rl',[]), ck.get('dl',[]), ck.get('cll',[]), ck.get('all_',[])
    print(f'Loaded {LOAD_MODEL}  best={best_dist:.3f}m  round={start_rnd}')
else:
    best_dist = -np.inf
    start_rnd = 0
    rl, dl, cll, all_ = [], [], [], []

best_w = None
print(f'ActorCritic params: {sum(p.numel() for p in ac.parameters()):,}')

ActorCritic params: 74,577


In [5]:
# ── PPO buffer + GAE + PPO update ─────────────────────────────────────────────

class PPOBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states    = []
        self.actions   = []
        self.log_probs = []
        self.rewards   = []
        self.values    = []
        self.dones     = []

    def push_episode(self, result: dict):
        """Append one worker's result dict into the buffer."""
        self.states.extend(result['states'])
        self.actions.extend(result['actions'])
        self.log_probs.extend(result['log_probs'])
        self.rewards.extend(result['rewards'])
        self.values.extend(result['values'])
        self.dones.extend(result['dones'])

    def __len__(self):
        return len(self.states)

    def compute_gae(self, last_values: list):
        """
        GAE over the full buffer.
        last_values: list of bootstrap values, one per episode in order they were appended.
        We treat each episode boundary (done=1) as a terminal, and handle bootstrap
        at the true episode end via the stored dones.
        For simplicity we compute a single GAE pass over the concatenated buffer
        using dones to reset the advantage at episode boundaries — standard practice.
        The final last_value used for bootstrapping is the mean of worker last_values.
        """
        T       = len(self.rewards)
        adv     = np.zeros(T, dtype=np.float32)
        ret     = np.zeros(T, dtype=np.float32)
        last_v  = np.mean(last_values)  # average bootstrap across workers
        values  = np.array(self.values + [last_v], dtype=np.float32)
        rewards = np.array(self.rewards, dtype=np.float32)
        dones   = np.array(self.dones,   dtype=np.float32)
        gae = 0.0
        for t in reversed(range(T)):
            delta  = rewards[t] + GAMMA * values[t+1] * (1-dones[t]) - values[t]
            gae    = delta + GAMMA * GAE_LAMBDA * (1-dones[t]) * gae
            adv[t] = gae
            ret[t] = adv[t] + values[t]
        return ret, adv

    def get_tensors(self, last_values):
        ret, adv = self.compute_gae(last_values)
        S   = torch.FloatTensor(np.array(self.states)).to(device)
        A   = torch.FloatTensor(np.array(self.actions)).to(device)
        LP  = torch.FloatTensor(np.array(self.log_probs)).to(device)
        R   = torch.FloatTensor(ret).to(device)
        Adv = torch.FloatTensor(adv).to(device)
        Adv = (Adv - Adv.mean()) / (Adv.std() + 1e-8)
        return S, A, LP, R, Adv


def update_ppo(buf: PPOBuffer, last_values: list):
    S, A, LP_old, R, Adv = buf.get_tensors(last_values)
    N = S.shape[0]
    total_pi = total_vf = n_up = 0
    for _ in range(PPO_EPOCHS):
        idx = torch.randperm(N, device=device)
        for start in range(0, N, MINIBATCH_SIZE):
            mb = idx[start:start+MINIBATCH_SIZE]
            lp_new, entropy, v_new = ac.evaluate_actions(S[mb], A[mb])
            ratio  = (lp_new - LP_old[mb]).exp()
            surr1  = ratio * Adv[mb]
            surr2  = ratio.clamp(1-CLIP_EPS, 1+CLIP_EPS) * Adv[mb]
            pi_loss = -torch.min(surr1, surr2).mean()
            vf_loss = F.mse_loss(v_new, R[mb])
            loss = pi_loss + VF_COEF * vf_loss - ENTROPY_COEF * entropy.mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(ac.parameters(), MAX_GRAD)
            opt.step()
            total_pi += pi_loss.item()
            total_vf += vf_loss.item()
            n_up     += 1
    buf.clear()
    return total_vf / max(n_up,1), total_pi / max(n_up,1)


ppo_buf = PPOBuffer()

In [6]:
# ── Eval rollout (runs in main process, deterministic, with optional render) ───

def orientation():
    qw, qx, qy, qz = mjdata.qpos[3:7]
    roll  = math.atan2(2*(qw*qx + qy*qz), 1 - 2*(qx**2 + qy**2))
    pitch = math.asin(float(np.clip(2*(qw*qy - qz*qx), -1, 1)))
    yaw   = math.atan2(2*(qw*qz + qx*qy), 1 - 2*(qy**2 + qz**2))
    return roll, pitch, yaw


def build_state(delta):
    return np.concatenate([
        mjdata.qpos[7:15].astype(np.float32),
        mjdata.qvel[6:14].astype(np.float32),
        delta.astype(np.float32),
    ])


def apply_ddelta(ctrl, delta, ddelta):
    delta_new = np.clip(delta + ddelta * MAX_DDELTA, -DELTA_LIMIT, DELTA_LIMIT)
    ctrl_new  = np.clip(ctrl  + delta_new, JOINT_LO, JOINT_HI)
    return ctrl_new, delta_new


def reset_sim():
    ip = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
    mujoco.mj_resetData(mjmodel, mjdata)
    mjdata.qpos[7:15] = ip
    mjdata.qpos[2]    = 0.10
    mujoco.mj_forward(mjmodel, mjdata)
    for _ in range(int(SETTLE_TIME / mjmodel.opt.timestep)):
        mjdata.ctrl[:] = ip
        mujoco.mj_step(mjmodel, mjdata)
    return ip.copy(), np.zeros(ACTION_DIM, dtype=np.float32)


def rollout_eval(render=False, fps=60, record_traj=False):
    ctrl, delta = reset_sim()
    x0          = mjdata.qpos[0]
    frames      = []
    traj_ctrl   = []
    traj_delta  = []
    renderer    = None

    if render:
        renderer = mujoco.Renderer(mjmodel, height=720, width=1280)
        cam = mujoco.MjvCamera()
        mujoco.mjv_defaultFreeCamera(mjmodel, cam)
        cam.distance  = 0.6
        cam.elevation = -20
        fe = max(1, int(1 / (fps * CTRL_DT)))

    s = build_state(delta)
    for t in range(int(EPISODE_DUR / CTRL_DT)):
        with torch.no_grad():
            st = torch.FloatTensor(s).unsqueeze(0).to(device)
            action, _, _ = ac.get_action(st, deterministic=True)
            an = action.squeeze(0).cpu().numpy()
        ctrl, delta = apply_ddelta(ctrl, delta, an)
        mjdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(mjmodel, mjdata)
        if record_traj:
            traj_ctrl.append(ctrl.copy())
            traj_delta.append(delta.copy())
        s = build_state(delta)
        if render and t % fe == 0:
            cam.lookat[0] = mjdata.qpos[0]
            cam.lookat[1] = mjdata.qpos[1]
            cam.azimuth   = (cam.azimuth + 0.2) % 360
            renderer.update_scene(mjdata, cam)
            frames.append(renderer.render().copy())
        _, p, _ = orientation()
        if mjdata.qpos[2] < 0.03 or abs(p) > math.radians(45):
            break

    dist = mjdata.qpos[0] - x0
    if renderer: renderer.close()
    if record_traj:
        return dist, frames, np.array(traj_ctrl, dtype=np.float32), np.array(traj_delta, dtype=np.float32)
    return dist, frames

## 🎬 Cinematic render (sim2real video)

`render_cinematic()` re-runs the deterministic eval episode and writes finished
footage: `dog_cinematic_<look>.xml` (same physics, new sky / floor / light rig),
a 4-shot camera choreography with damped follow, motion blur from control-rate→fps
resampling, and a bloom + vignette + contrast grade.

Two looks: **`night`** (dark studio, glossy cyan grid floor) and **`day`**
(blue sky, matte concrete slabs).

```python
render_cinematic('movies/albert_night.mp4', weights=best_w, look='night')
render_cinematic('movies/albert_day.mp4',   weights=best_w, look='day')
render_cinematic('movies/albert_slowmo.mp4', weights=best_w, slowmo=0.4)

# same angle across training checkpoints → watch it learn to walk
render_progress_montage('movies/albert_learning.mp4', look='day', n_clips=8)
```


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
#  CINEMATIC RENDERER — sim2real footage (YouTube)
#  Physics untouched: dog_cinematic_<look>.xml is dog.xml with new visuals only
#  (skybox / floor / lights).  Two looks:
#     'night' → dark tech studio, glossy grid floor, warm key + blue rim
#     'day'   → blue sky, matte concrete slab floor, sun + sky fill
#  Plus: shot list, damped follow camera, motion blur, bloom/vignette grade,
#  and render_progress_montage() to show the policy while it is still learning.
# ══════════════════════════════════════════════════════════════════════════════
import re, glob

# ── Procedural concrete texture for the 'day' floor ──────────────────────────
def _value_noise(n, res, rng):
    """Tileable smooth value noise at resolution `res`, sampled to n x n."""
    g = rng.random((res, res)).astype(np.float32)
    g = np.concatenate([g, g[:1]], 0)
    g = np.concatenate([g, g[:, :1]], 1)
    x = np.linspace(0, res, n, endpoint=False)
    i = x.astype(int)
    f = (x - i)[:, None]
    f = f * f * (3 - 2 * f)                                   # smoothstep
    a, b = g[i][:, i], g[i][:, i + 1]
    c, d = g[i + 1][:, i], g[i + 1][:, i + 1]
    top, bot = a + (b - a) * f.T, c + (d - c) * f.T
    return top + (bot - top) * f


def make_floor_texture(path='textures/floor_day.png', n=1024, seed=7):
    """Seamless concrete slab texture (fbm stains + grain + saw-cut seams)."""
    if os.path.exists(path):
        return path
    from PIL import Image
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    rng = np.random.default_rng(seed)
    fbm = sum(_value_noise(n, r, rng) * a
              for r, a in [(4, .5), (8, .25), (16, .15), (32, .08), (64, .05)])
    fbm = (fbm - fbm.min()) / (np.ptp(fbm) + 1e-9)
    v = 0.46 + 0.15 * (fbm - 0.5) + rng.normal(0, .012, (n, n)).astype(np.float32)
    u = np.arange(n) / n
    seam = (np.abs(((u * 2 + .5) % 1) - .5) < 0.004).astype(np.float32)
    v = np.clip(v - 0.075 * (seam[None, :] + seam[:, None]), 0, 1)
    rgb = np.stack([v * 1.00, v * 0.985, v * 0.955], -1)      # warm grey
    Image.fromarray((rgb * 255).astype(np.uint8)).save(path)
    print(f'  wrote {path}')
    return path


# ── The two looks: <visual>, <asset> and lights+floor blocks ─────────────────
_NIGHT = dict(
    visual='''  <visual>
    <global offwidth="1920" offheight="1080"/>
    <quality shadowsize="16384" offsamples="8" numslices="64" numstacks="32" numquads="20"/>
    <map shadowscale="0.5" shadowclip="1.0" znear="0.004" zfar="60" haze="0.55"/>
    <headlight ambient="0.22 0.24 0.32" diffuse="0.10 0.10 0.13" specular="0.04 0.04 0.05"/>
    <rgba haze="0.20 0.28 0.42 1"/>
  </visual>
''',
    asset='''    <!-- Cielo: gradiente notte -> blu crepuscolo all'orizzonte -->
    <texture name="skybox" type="skybox" builtin="gradient"
             rgb1="0.010 0.014 0.035" rgb2="0.20 0.28 0.42"
             markrgb="0.90 0.93 1.00" mark="random" random="0.00004"
             width="1024" height="1024"/>

    <!-- Pavimento: checker scuro + griglia ciano, lucido -->
    <texture name="grid" type="2d" builtin="checker"
             rgb1="0.125 0.138 0.165" rgb2="0.165 0.182 0.215"
             markrgb="0.20 0.60 0.85" mark="edge"
             width="1024" height="1024"/>
    <material name="floor_mat" texture="grid"
              texrepeat="16 16" texuniform="true"
              reflectance="0.35" shininess="0.7" specular="0.35"/>
''',
    lights='''    <!-- NB: gli spot bassi e radenti anneriscono il pavimento lontano nel
         renderer MuJoCo -> rim/fill sono direzionali, solo la key e' uno spot. -->
    <light name="ambient_sun" directional="true" pos="0 0 5" dir="-0.25 -0.30 -1"
           diffuse="0.90 0.94 1.05" specular="0.06 0.06 0.07" castshadow="false"/>
    <light name="key" mode="trackcom" pos="0.35 0.30 0.60" dir="-0.45 -0.40 -1"
           cutoff="45" exponent="3" attenuation="1 0 0"
           diffuse="1.15 1.09 0.99" specular="0.55 0.52 0.46" castshadow="true"/>
    <light name="rim" directional="true" pos="-3 -2 1.2" dir="0.78 0.52 -0.35"
           diffuse="0.30 0.46 0.75" specular="0.22 0.34 0.55" castshadow="false"/>
    <light name="fill" directional="true" pos="-0.5 3 1.5" dir="0.12 -0.75 -0.65"
           diffuse="0.12 0.13 0.19" specular="0.03 0.03 0.04" castshadow="false"/>

    <!-- Pavimento grande: nessun bordo visibile in campo -->
    <geom name="floor" type="plane" size="25 25 0.05"
          material="floor_mat" condim="3"/>
''',
    grade=dict(bloom=0.30, thr=0.72, sat=1.10, contrast=1.06, vignette=0.30),
)

_DAY = dict(
    visual='''  <visual>
    <global offwidth="1920" offheight="1080"/>
    <quality shadowsize="16384" offsamples="8" numslices="64" numstacks="32" numquads="20"/>
    <map shadowscale="0.5" shadowclip="1.0" znear="0.004" zfar="60" haze="0.60"/>
    <headlight ambient="0.30 0.33 0.40" diffuse="0.10 0.10 0.11" specular="0.05 0.05 0.05"/>
    <rgba haze="0.80 0.87 0.95 1"/>
  </visual>
''',
    asset='''    <!-- Cielo diurno -->
    <texture name="skybox" type="skybox" builtin="gradient"
             rgb1="0.24 0.45 0.78" rgb2="0.86 0.92 0.98"
             width="1024" height="1024"/>

    <!-- Pavimento: lastre di cemento (texture procedurale) -->
    <texture name="grid" type="2d" file="textures/floor_day.png"/>
    <material name="floor_mat" texture="grid"
              texrepeat="3 3" texuniform="true"
              reflectance="0.08" shininess="0.25" specular="0.18"/>
''',
    lights='''    <light name="sun" directional="true" pos="0 0 5" dir="-0.35 -0.42 -1"
           diffuse="0.95 0.91 0.83" specular="0.35 0.34 0.30" castshadow="true"/>
    <light name="sky" directional="true" pos="0 0 5" dir="0.30 0.45 -1"
           diffuse="0.22 0.26 0.34" specular="0.05 0.05 0.06" castshadow="false"/>

    <geom name="floor" type="plane" size="25 25 0.05"
          material="floor_mat" condim="3"/>
''',
    grade=dict(bloom=0.16, thr=0.85, sat=1.07, contrast=1.05, vignette=0.22),
)

CINE_LOOKS = {'night': _NIGHT, 'day': _DAY}


def build_cinematic_xml(look='night', src='dog.xml', dst=None):
    """dog.xml with film visuals. Same bodies/joints/actuators → same physics."""
    L   = CINE_LOOKS[look]
    dst = dst or f'dog_cinematic_{look}.xml'
    if look == 'day':
        make_floor_texture()
    x = open(src).read()
    subs = [
        (r'  <visual>.*?</visual>\n',                                      L['visual']),
        (r'    <!-- Skybox -->.*?texrepeat="6 6" reflectance="0\.15"/>\n',  L['asset']),
        (r'    <!-- Luci -->.*?material="floor_mat" condim="3"/>\n',        L['lights']),
    ]
    for pat, rep in subs:
        x, n = re.subn(pat, rep, x, flags=re.S)
        assert n == 1, f'cinematic patch failed for {pat[:30]!r} (n={n})'
    with open(dst, 'w') as f:
        f.write(x)
    return os.path.abspath(dst)


# ── Colour grade: bloom + saturation + S-contrast + vignette ─────────────────
def _blur(a, iters):
    for _ in range(iters):
        a = (a + np.roll(a, 1, 0) + np.roll(a, -1, 0) + np.roll(a, 1, 1) + np.roll(a, -1, 1)) / 5.0
    return a


def cine_grade(img, bloom=0.30, thr=0.72, sat=1.10, contrast=1.06, vignette=0.30):
    f = img.astype(np.float32) / 255.0
    small = f[::4, ::4]                                  # bloom at 1/4 res
    b = _blur(np.clip(small - thr, 0, None) / max(1e-6, 1 - thr), 10)
    b = np.repeat(np.repeat(b, 4, 0), 4, 1)[:f.shape[0], :f.shape[1]]
    f = f + bloom * b
    m = f.mean(2, keepdims=True)
    f = m + (f - m) * sat
    f = np.clip((f - 0.5) * contrast + 0.5, 0, 1)
    h, w = f.shape[:2]
    yy = np.linspace(-1, 1, h)[:, None]
    xx = np.linspace(-1, 1, w)[None, :]
    r = np.sqrt(xx ** 2 * 0.85 + yy ** 2)
    f *= (1.0 - vignette * np.clip((r - 0.55) / 0.85, 0, 1) ** 2)[..., None]
    return (np.clip(f, 0, 1) * 255).astype(np.uint8)


# ── Shot lists: (duration s, azimuth, elevation, distance, lookat height) ────
# az/el/dist are (start, end), eased with a smoothstep inside each shot.
CINE_SHOTS = [
    dict(dur=2.4, az=(196, 228), el=(-8,  -13), dist=(0.95, 0.58), h=0.090),  # wide reveal, dolly in
    dict(dur=1.8, az=(118, 100), el=(-5,   -7), dist=(0.46, 0.38), h=0.068),  # low side, close
    dict(dur=1.8, az=(305, 332), el=(-24, -19), dist=(0.62, 0.44), h=0.085),  # high 3/4 push
    dict(dur=1e9, az=(212, 252), el=(-13, -16), dist=(0.72, 0.60), h=0.090),  # long orbit tail
]
# Single continuous 3/4 orbit — used for the training-progress montage so every
# clip is shot from the same angle and only the gait changes.
CINE_SHOTS_ORBIT = [dict(dur=1e9, az=(216, 258), el=(-13, -17), dist=(0.70, 0.55), h=0.088)]

_smooth = lambda u: u * u * (3 - 2 * u)


def _cine_cam(cam, t, shots, look_at):
    """Place the free camera for take-time t (s) from the shot list."""
    t0 = 0.0
    for sh in shots:
        if t < t0 + sh['dur'] or sh is shots[-1]:
            u = _smooth(min(1.0, (t - t0) / sh['dur']))
            lerp = lambda ab: ab[0] + (ab[1] - ab[0]) * u
            cam.azimuth   = lerp(sh['az'])
            cam.elevation = lerp(sh['el'])
            cam.distance  = lerp(sh['dist'])
            cam.lookat[:] = [look_at[0], look_at[1], sh['h']]
            return
        t0 += sh['dur']


# ── Caption overlay (used by the montage) ────────────────────────────────────
_FONT_CANDIDATES = ['/System/Library/Fonts/Helvetica.ttc',
                    '/System/Library/Fonts/Supplemental/Arial Bold.ttf',
                    '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf']


def _font(size):
    from PIL import ImageFont
    for p in _FONT_CANDIDATES:
        try:
            return ImageFont.truetype(p, size)
        except Exception:
            pass
    return ImageFont.load_default()


def _caption(frame, title, subtitle='', progress=None):
    """Burn a title / subtitle / progress bar into the bottom-left of a frame."""
    from PIL import Image, ImageDraw
    im = Image.fromarray(frame)
    d  = ImageDraw.Draw(im, 'RGBA')
    W, H = im.size
    s  = H / 1080.0
    x, y = int(64 * s), int(H - 150 * s)
    d.rectangle([0, int(H - 210 * s), W, H], fill=(0, 0, 0, 90))
    d.text((x, y), title, font=_font(int(46 * s)), fill=(255, 255, 255, 235))
    if subtitle:
        d.text((x, y + int(56 * s)), subtitle, font=_font(int(30 * s)), fill=(150, 210, 255, 220))
    if progress is not None:
        bw, bh = W - 2 * x, int(6 * s)
        by = int(H - 40 * s)
        d.rectangle([x, by, x + bw, by + bh], fill=(255, 255, 255, 45))
        d.rectangle([x, by, x + int(bw * float(np.clip(progress, 0, 1))), by + bh],
                    fill=(90, 200, 255, 230))
    return np.asarray(im)


# ── One take: rollout + camera + motion blur, streamed into a VideoWriter ────
def _take(vid, weights=None, look='night', shots=None, duration=None, fps=60,
          slowmo=1.0, grade=True, overlay=None, cmodel=None, cdata=None,
          rend=None, width=1920, height=1080):
    """Render one deterministic episode into an open mediapy VideoWriter.

    overlay(frame, t, dist) -> frame  is applied after the grade.
    Returns (distance_m, n_frames_written).
    """
    shots = shots if shots is not None else CINE_SHOTS
    dur   = duration if duration is not None else EPISODE_DUR
    gkw   = CINE_LOOKS[look]['grade']

    if weights is not None:
        saved = {k: v.clone() for k, v in ac.state_dict().items()}
        ac.load_state_dict({k: v.to(device) for k, v in weights.items()})

    opt = mujoco.MjvOption()
    opt.geomgroup[3] = 0                       # hide collision capsules
    for g in range(len(opt.sitegroup)):
        opt.sitegroup[g] = 0                   # hide IMU / foot sites
    cam = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(cmodel, cam)

    ip = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
    mujoco.mj_resetData(cmodel, cdata)
    cdata.qpos[7:15] = ip
    cdata.qpos[2]    = 0.10
    mujoco.mj_forward(cmodel, cdata)
    for _ in range(int(SETTLE_TIME / cmodel.opt.timestep)):
        cdata.ctrl[:] = ip
        mujoco.mj_step(cmodel, cdata)

    ctrl, delta = ip.copy(), np.zeros(ACTION_DIM, dtype=np.float32)
    x0   = cdata.qpos[0]
    look_at = np.array([cdata.qpos[0], cdata.qpos[1]], dtype=np.float64)

    def _shoot(t):
        _cine_cam(cam, t, shots, look_at)
        rend.update_scene(cdata, cam, opt)
        F = mujoco.mjtRndFlag
        for flag in (F.mjRND_SHADOW, F.mjRND_REFLECTION, F.mjRND_SKYBOX, F.mjRND_HAZE):
            rend.scene.flags[flag] = 1
        rend.update_scene(cdata, cam, opt)
        return rend.render().astype(np.float32)

    def _emit(fr, t):
        fr = cine_grade(fr, **gkw) if grade else fr.astype(np.uint8)
        if overlay is not None:
            fr = overlay(fr, t, float(cdata.qpos[0] - x0))
        vid.add_image(fr)

    n_out, t_sim = 0, 0.0
    prev_f, prev_t = _shoot(0.0), 0.0
    dt_out = slowmo / fps                      # sim seconds per output frame
    _emit(prev_f, 0.0); n_out += 1

    for _ in range(int(dur / CTRL_DT)):
        with torch.no_grad():
            s  = np.concatenate([cdata.qpos[7:15].astype(np.float32),
                                 cdata.qvel[6:14].astype(np.float32),
                                 delta.astype(np.float32)])
            a, _, _ = ac.get_action(torch.FloatTensor(s).unsqueeze(0).to(device),
                                    deterministic=True)
            an = a.squeeze(0).cpu().numpy()
        delta = np.clip(delta + an * MAX_DDELTA, -DELTA_LIMIT, DELTA_LIMIT)
        ctrl  = np.clip(ctrl + delta, JOINT_LO, JOINT_HI)      # same law as training
        cdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(cmodel, cdata)
        t_sim += CTRL_DT

        # damped follow: the camera lags the body slightly → organic motion
        look_at += (np.array([cdata.qpos[0], cdata.qpos[1]]) - look_at) * 0.15

        cur_f = _shoot(t_sim)
        # resample control rate → fps, blending neighbours = motion blur
        while n_out * dt_out <= t_sim + 1e-9:
            w  = (n_out * dt_out - prev_t) / max(1e-9, t_sim - prev_t)
            _emit(prev_f + (cur_f - prev_f) * np.clip(w, 0.0, 1.0), n_out * dt_out)
            n_out += 1
        prev_f, prev_t = cur_f, t_sim

        qw, qx, qy, qz = cdata.qpos[3:7]
        pitch = math.asin(float(np.clip(2 * (qw * qy - qz * qx), -1, 1)))
        if cdata.qpos[2] < 0.03 or abs(pitch) > math.radians(45):
            break                                              # fell over

    if weights is not None:
        ac.load_state_dict(saved)
    return float(cdata.qpos[0] - x0), n_out


def _scene(look, width, height):
    xml = build_cinematic_xml(look)
    m   = mujoco.MjModel.from_xml_path(xml)
    return m, mujoco.MjData(m), mujoco.Renderer(m, height=height, width=width)


def render_cinematic(out='movies/albert_cinematic.mp4', weights=None, look='night',
                     width=1920, height=1080, fps=60, slowmo=1.0, duration=None,
                     shots=None, grade=True, qp=17, verbose=True):
    """One deterministic eval episode rendered as finished footage.

    look='night' | 'day'      slowmo=0.4 → 2.5x slower.  Returns (dist_m, frames).
    """
    cmodel, cdata, rend = _scene(look, width, height)
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
    with media.VideoWriter(out, shape=(height, width), fps=fps, qp=qp) as vid:
        dist, n = _take(vid, weights=weights, look=look, shots=shots, duration=duration,
                        fps=fps, slowmo=slowmo, grade=grade,
                        cmodel=cmodel, cdata=cdata, rend=rend)
    rend.close()
    if verbose:
        print(f'  → {out}   {n} frames @ {fps}fps   dist={dist:.3f} m   [{look}]')
    return dist, n


def render_progress_montage(out='movies/albert_learning.mp4', look='day',
                            checkpoints=None, n_clips=6, seconds_each=3.0,
                            pattern='models/parallel_ppo_rnd*.pth', key='ac',
                            width=1920, height=1080, fps=60, slowmo=1.0,
                            shots=None, qp=17, verbose=True):
    """Montage of the SAME shot across training checkpoints: watch it learn.

    Picks `n_clips` evenly spaced checkpoints matching `pattern` (or use
    `checkpoints=[paths]`), renders `seconds_each` of each and burns in a
    'round N' caption + progress bar.  key='ac' = policy as it was at that
    round; key='best_w' = best policy so far.
    """
    if checkpoints is None:
        found = sorted(glob.glob(pattern),
                       key=lambda p: int(re.search(r'rnd(\d+)', p).group(1)))
        if not found:
            raise FileNotFoundError(f'no checkpoints matching {pattern}')
        idx = np.linspace(0, len(found) - 1, min(n_clips, len(found))).round().astype(int)
        checkpoints = [found[i] for i in dict.fromkeys(idx)]

    shots = shots if shots is not None else CINE_SHOTS_ORBIT
    cmodel, cdata, rend = _scene(look, width, height)
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)

    with media.VideoWriter(out, shape=(height, width), fps=fps, qp=qp) as vid:
        for i, path in enumerate(checkpoints):
            ck  = torch.load(path, map_location='cpu')
            rnd = int(re.search(r'rnd(\d+)', path).group(1))
            w   = ck.get(key) or ck['ac']
            prog = (i + 1) / len(checkpoints)
            cap = lambda fr, t, dist: _caption(
                fr, f'round {rnd}',
                f'{rnd * N_WORKERS} episodes  ·  {dist:+.2f} m', prog)
            dist, n = _take(vid, weights=w, look=look, shots=shots,
                            duration=seconds_each * slowmo, fps=fps, slowmo=slowmo,
                            overlay=cap, cmodel=cmodel, cdata=cdata, rend=rend)
            if verbose:
                print(f'  clip {i+1}/{len(checkpoints)}  round {rnd:<5d} '
                      f'dist={dist:+.3f} m  ({n} frames)')
    rend.close()
    if verbose:
        print(f'  → {out}   [{look}]')
    return out


In [8]:
# ══════════════════════════════════════════════════════════════
#  Trajectory + policy export  (same output as AllLegs_def)
# ══════════════════════════════════════════════════════════════

def export_trajectory(traj: np.ndarray, tag: str = "best", delta_traj: np.ndarray = None):
    T    = traj.shape[0]
    base = f"trajectories/{tag}"

    np.save(f"{base}.npy", traj)
    print(f"  Saved {base}.npy  shape={traj.shape}")

    if delta_traj is not None:
        np.save(f"{base}_delta.npy", delta_traj)
        print(f"  Saved {base}_delta.npy  (delta buffer evolution)")

    header = "step_ms," + ",".join(JOINT_NAMES)
    rows = [f"{i*CTRL_DT*1000:.2f}," + ",".join(f"{v:.6f}" for v in row)
            for i, row in enumerate(traj)]
    with open(f"{base}.csv", "w") as f:
        f.write(header + "\n" + "\n".join(rows) + "\n")
    print(f"  Saved {base}.csv  ({T} rows)")

    scale = 10_000
    # Store as offset from neutral pose (OFFSETS) so the ESP32 only needs
    # to add the relative delta to CENTER — no absolute angle arithmetic.
    #   rel[step][j] = traj[step][j] - OFFSETS[j]
    #   on ESP32:  pulse = CENTER + rel * PULSE_PER_RAD * JOINT_DIR
    traj_rel = traj - OFFSETS[np.newaxis, :]        # shape (T, 8), centred at 0
    int16 = np.clip(np.round(traj_rel * scale), -32768, 32767).astype(np.int16)

    # Sanity: at step 0 (settled neutral) all values should be near zero
    max_abs = np.abs(traj_rel[0]).max()
    if max_abs > 0.05:
        print(f"  Warning: step-0 max |offset| = {max_abs:.4f} rad "
              f"(robot may not start at neutral pose)")

    lines = [
        f"/* Auto-generated trajectory  tag={tag}",
        f" * Steps: {T}  dt: {CTRL_DT*1000:.2f} ms  ({1/CTRL_DT:.0f} Hz)",
        f" * Joints: {', '.join(JOINT_NAMES)}",
        f" * Control scheme: delta-delta (ΔΔθ) — acceleration-level actions",
        f" * VALUES ARE RELATIVE TO NEUTRAL POSE (OFFSETS subtracted):",
        f" * neutral = [{', '.join(f'{v:.4f}' for v in OFFSETS)}] rad",
        f" * angle_rad  = OFFSET[j] + traj[step][j] / {scale}.0f",
        f" * pulse      = CENTER    + traj[step][j] / {scale}.0f * PULSE_PER_RAD * DIR[ch]",
        " */",
        "#pragma once", "#include <stdint.h>", "",
        f"#define TRAJ_STEPS  {T}",
        f"#define TRAJ_JOINTS 8",
        f"#define TRAJ_DT_MS  {int(round(CTRL_DT * 1000))}",
        f"#define TRAJ_SCALE  {scale}",
        f"/* neutral offsets (rad×TRAJ_SCALE) for reference:",
        f"   HIP_INIT  = {int(round(HIP_INIT  * scale))}",
        f"   KNEE_INIT = {int(round(KNEE_INIT * scale))} */", "",
        f"static const int16_t traj[{T}][8] = {{",
    ]
    for i, row in enumerate(int16):
        vals  = ", ".join(f"{int(v):6d}" for v in row)
        comma = "," if i < T - 1 else " "
        lines.append(f"  {{ {vals} }}{comma}  /* t={i*CTRL_DT*1000:7.1f} ms */")
    lines.append("};")
    with open(f"{base}.h", "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"  Saved {base}.h  ({T*8*2/1024:.1f} KB)")

    t_axis = np.arange(T) * CTRL_DT
    ncols  = 3 if delta_traj is not None else 2
    fig, axes = plt.subplots(4, ncols, figsize=(6*ncols, 10), sharex=True)

    for idx, name in enumerate(JOINT_NAMES):
        ax_pos = axes[idx // 2, (idx % 2) * (1 if ncols == 2 else 1)]
        ax_pos.plot(t_axis, traj[:, idx], lw=1.2)
        ax_pos.set_title(f"{name} pos", fontsize=9)
        ax_pos.set_ylabel("rad", fontsize=7)
        ax_pos.grid(True, alpha=0.3)

    if delta_traj is not None:
        for idx, name in enumerate(JOINT_NAMES):
            ax_d = axes[idx // 2, 2]
            ax_d.plot(t_axis, delta_traj[:, idx], lw=1.0, alpha=0.7, label=name)
        for r in range(4):
            axes[r, 2].legend(fontsize=6)
            axes[r, 2].set_title("delta buffer", fontsize=9)
            axes[r, 2].grid(True, alpha=0.3)

    for c in range(ncols):
        axes[-1, c].set_xlabel("time (s)")
    plt.suptitle(f"Trajectory (ΔΔθ) — {T} steps @ {1/CTRL_DT:.0f} Hz", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{base}_plot.png", dpi=120)
    plt.close()
    print(f"  Saved {base}_plot.png")


# ══════════════════════════════════════════════════════════════
#  ESP32 policy export
#  Writes a self-contained C header with:
#    • all actor weights/biases as float arrays
#    • all control constants (limits, deltas, offsets)
#    • a complete policy_step() function ready to #include
# ══════════════════════════════════════════════════════════════
def export_policy_header(path: str = "models/policy.h"):
    """Dump the current actor weights + a full C inference routine.

    On the ESP32 call once per control tick:
        policy_step(state, ctrl, delta);   // updates ctrl[] and delta[] in-place
    """
    sd = ac.state_dict()

    # actor weights  (pi.0 = first Linear, pi.2 = second Linear)
    W1 = sd["pi.0.weight"].cpu().numpy()   # (HIDDEN_DIM, STATE_DIM)
    b1 = sd["pi.0.bias"  ].cpu().numpy()   # (HIDDEN_DIM,)
    W2 = sd["pi.2.weight"].cpu().numpy()   # (ACTION_DIM, HIDDEN_DIM)
    b2 = sd["pi.2.bias"  ].cpu().numpy()   # (ACTION_DIM,)

    H = W1.shape[0]   # HIDDEN_DIM
    S = W1.shape[1]   # STATE_DIM
    A = W2.shape[0]   # ACTION_DIM

    def arr1d(name, vec):
        vals = ", ".join(f"{v:.8f}f" for v in vec.flatten())
        return f"static const float {name}[{vec.size}] = {{{vals}}};\n"

    def arr2d(name, mat, comment=""):
        rows, cols = mat.shape
        out = [f"/* {comment} */" if comment else "",
               f"static const float {name}[{rows}][{cols}] = {{"]
        for i, row in enumerate(mat):
            vals  = ", ".join(f"{v:.8f}f" for v in row)
            comma = "," if i < rows - 1 else ""
            out.append(f"  {{ {vals} }}{comma}")
        out.append("};\n")
        return "\n".join(out) + "\n"

    def flt(name, val, comment=""):
        c = f"  /* {comment} */" if comment else ""
        return f"static const float {name} = {val:.8f}f;{c}\n"

    lines = []
    lines.append(f"""\
#pragma once
/*
 * Auto-generated policy header — DO NOT EDIT BY HAND
 *
 * Architecture : {S}-input  ->  ReLU({H})  ->  tanh({A})-output
 * Training     : PPO  delta-delta (acceleration-level) control
 * Target speed : {TARGET_VX:.4f} m/s  ({1.0/TARGET_VX:.1f} s per metre)
 * Control dt   : {CTRL_DT*1000:.2f} ms  ({1/CTRL_DT:.0f} Hz)
 *
 * Flash footprint:
 *   W1 {H}x{S} = {H*S*4} bytes
 *   W2 {A}x{H} = {A*H*4} bytes
 *   total approx {(H*S + H + A*H + A)*4} bytes
 *
 * Quick-start (Arduino/ESP-IDF):
 *
 *   #include "policy.h"
 *
 *   float ctrl[POLICY_ACTION_DIM]  = {{HIP0, KNEE0, HIP0, KNEE0,
 *                                      HIP0, KNEE0, HIP0, KNEE0}};
 *   float delta[POLICY_ACTION_DIM] = {{0}};
 *
 *   void loop() {{
 *     float state[POLICY_STATE_DIM];
 *     read_encoders(state);          // jpos[0..7]
 *     read_imu_jvel(state + 8);      // jvel[0..7]
 *     memcpy(state + 16, delta, sizeof(delta));  // current delta buffer
 *     policy_step(state, ctrl, delta);
 *     send_joint_targets(ctrl);
 *     delay(POLICY_CTRL_DT_MS);
 *   }}
 */

#include <math.h>
#include <string.h>

#define POLICY_STATE_DIM  {S}
#define POLICY_HIDDEN_DIM {H}
#define POLICY_ACTION_DIM {A}
#define POLICY_CTRL_DT_MS {int(round(CTRL_DT * 1000))}

#define POLICY_HIP_INIT   {HIP_INIT:.6f}f
#define POLICY_KNEE_INIT  {KNEE_INIT:.6f}f

""")

    lines.append("/* ---- Layer 1  W1[HIDDEN][STATE] ---- */\n")
    lines.append(arr2d("POLICY_W1", W1, f"W1[{H}][{S}]: weights for hidden layer"))
    lines.append("/* ---- Layer 1  b1[HIDDEN] ---- */\n")
    lines.append(arr1d("POLICY_b1", b1))
    lines.append("\n/* ---- Layer 2  W2[ACTION][HIDDEN] ---- */\n")
    lines.append(arr2d("POLICY_W2", W2, f"W2[{A}][{H}]: weights for output layer"))
    lines.append("/* ---- Layer 2  b2[ACTION] ---- */\n")
    lines.append(arr1d("POLICY_b2", b2))

    lines.append("\n/* ---- Control constants ---- */\n")
    lines.append(flt("POLICY_MAX_DDELTA",  MAX_DDELTA,  "acceleration scale (rad/step)"))
    lines.append(flt("POLICY_DELTA_LIMIT", DELTA_LIMIT, "max |delta| (rad/step)"))
    lines.append(flt("POLICY_TARGET_VX",   TARGET_VX,   "desired forward speed (m/s)"))

    lines.append(f"\nstatic const float POLICY_JOINT_LO[{A}] = {{"
                 + ", ".join(f"{v:.6f}f" for v in JOINT_LO) + "};\n")
    lines.append(f"static const float POLICY_JOINT_HI[{A}] = {{"
                 + ", ".join(f"{v:.6f}f" for v in JOINT_HI) + "};\n")
    lines.append(f"static const float POLICY_OFFSETS[{A}]   = {{"
                 + ", ".join(f"{v:.6f}f" for v in OFFSETS)   + "};\n")

    lines.append(f"""
/* ---- policy_step: run one control tick -------------------------
 *
 * state[{S}] : input  -- {{ jpos[8], jvel[8], delta[8] }}
 *   jpos  = joint angles (rad) from encoders, order: FL_hip, FL_knee,
 *           FR_hip, FR_knee, RL_hip, RL_knee, RR_hip, RR_knee
 *   jvel  = joint velocities (rad/s) from encoders / finite difference
 *   delta = persistent delta buffer — zero-initialise at boot, then pass
 *           the same array every tick so it carries state between calls
 *
 * ctrl[{A}]  : in/out -- joint position targets (rad); send to servos after call
 * delta[{A}] : in/out -- persistent buffer, updated in-place; DO NOT reset between ticks
 *
 * Computation:
 *   h[i]      = ReLU( sum_j W1[i][j]*state[j] + b1[i] )
 *   mean[k]   = sum_i W2[k][i]*h[i] + b2[k]
 *   action[k] = tanh(mean[k])
 *   delta[k]  = clamp(delta[k] + action[k]*MAX_DDELTA, -DELTA_LIMIT, +DELTA_LIMIT)
 *   ctrl[k]   = clamp(ctrl[k]  + delta[k],              JOINT_LO[k],  JOINT_HI[k])
 */
static inline void policy_step(
    const float state[POLICY_STATE_DIM],
    float       ctrl [POLICY_ACTION_DIM],
    float       delta[POLICY_ACTION_DIM])
{{
    int i, j, k;

    /* --- hidden layer: h = ReLU(W1 @ state + b1) ------------ */
    float h[POLICY_HIDDEN_DIM];
    for (i = 0; i < POLICY_HIDDEN_DIM; i++) {{
        float acc = POLICY_b1[i];
        for (j = 0; j < POLICY_STATE_DIM; j++)
            acc += POLICY_W1[i][j] * state[j];
        h[i] = (acc > 0.0f) ? acc : 0.0f;
    }}

    /* --- output layer: mean = W2 @ h + b2, action = tanh ---- */
    float action[POLICY_ACTION_DIM];
    for (k = 0; k < POLICY_ACTION_DIM; k++) {{
        float acc = POLICY_b2[k];
        for (i = 0; i < POLICY_HIDDEN_DIM; i++)
            acc += POLICY_W2[k][i] * h[i];
        action[k] = tanhf(acc);
    }}

    /* --- two-level integration ------------------------------- */
    for (k = 0; k < POLICY_ACTION_DIM; k++) {{
        /* acceleration -> velocity */
        delta[k] += action[k] * POLICY_MAX_DDELTA;
        if      (delta[k] >  POLICY_DELTA_LIMIT) delta[k] =  POLICY_DELTA_LIMIT;
        else if (delta[k] < -POLICY_DELTA_LIMIT) delta[k] = -POLICY_DELTA_LIMIT;

        /* velocity -> position */
        ctrl[k] += delta[k];
        if      (ctrl[k] > POLICY_JOINT_HI[k]) ctrl[k] = POLICY_JOINT_HI[k];
        else if (ctrl[k] < POLICY_JOINT_LO[k]) ctrl[k] = POLICY_JOINT_LO[k];
    }}
}}
""")

    with open(path, "w") as f:
        f.writelines(lines)

    total_params = H*S + H + A*H + A
    print(f"  Saved {path}")
    print(f"  Actor: W1({H}x{S}) + b1({H}) + W2({A}x{H}) + b2({A}) "
          f"= {total_params} floats = {total_params*4/1024:.1f} KB")


In [9]:
# ── Parallel training loop ────────────────────────────────────────────────────
#
# Each round:
#   1. Serialize current weights to numpy (cheap, avoids pickle overhead of tensors)
#   2. Submit N_WORKERS futures — each runs one full episode in a subprocess
#   3. Collect results, push into shared PPOBuffer
#   4. When buffer >= N_STEPS_PER_UPDATE, run PPO update
#   5. Periodically save checkpoint and render eval video

from _parallel_worker import _worker

nm = 0
ppo_update_count = 0
t0 = time.time()

# Keep track of last_values across the batch for proper GAE bootstrap
pending_last_values = []

with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
    for rnd in trange(start_rnd, start_rnd + N_ROUNDS, desc='rounds'):

        # ── 1. Snapshot current weights as numpy (pickleable) ──────────────
        w_np = ac.weights_numpy()

        # ── 2. Launch N_WORKERS parallel episodes ───────────────────────────
        futures = [
            pool.submit(_worker, wid, w_np, CFG)
            for wid in range(N_WORKERS)
        ]

        # ── 3. Collect results + fill buffer ────────────────────────────────
        round_dists = []
        round_rs    = []
        last_values = []

        for fut in futures:
            res = fut.result()          # blocks until this worker is done
            ppo_buf.push_episode(res)
            last_values.append(res['last_value'])
            round_dists.append(res['dist'])
            round_rs.append(res['tot_r'])

        # Keep track of bootstrap values; they line up with the buffer segments
        pending_last_values.extend(last_values)

        avg_dist = float(np.mean(round_dists))
        avg_r    = float(np.mean(round_rs))
        dl.append(avg_dist)
        rl.append(avg_r)

        if avg_dist > best_dist:
            best_dist = avg_dist
            best_w = {k: v.clone() for k, v in ac.state_dict().items()}

        # ── 4. PPO update when buffer is full ────────────────────────────────
        cl = al = 0.0
        if len(ppo_buf) >= N_STEPS_PER_UPDATE:
            cl, al = update_ppo(ppo_buf, pending_last_values)
            pending_last_values = []
            ppo_update_count += 1
        cll.append(cl); all_.append(al)

        # ── 5. Logging ───────────────────────────────────────────────────────
        if rnd % 5 == 0:
            elapsed = time.time() - t0
            steps_per_sec = (rnd - start_rnd + 1) * N_WORKERS * int(EPISODE_DUR/CTRL_DT) / elapsed
            print(f'  rnd {rnd:4d} | dist={avg_dist:.3f}m avg10={np.mean(dl[-10:]):.3f} '
                  f'| R={avg_r:.1f} | buf={len(ppo_buf)} '
                  f'| {steps_per_sec:.0f} steps/s | updates={ppo_update_count}')

        # ── 6. Periodic eval render ──────────────────────────────────────────
        if rnd % RENDER_EVERY == 0 and rnd > start_rnd and best_w:
            train_w = {k: v.clone() for k, v in ac.state_dict().items()}
            ac.load_state_dict(best_w)
            bd, frames = rollout_eval(render=True)
            if frames:
                out = f'movies/parallel_ppo_{nm:03d}_rnd{rnd:04d}_{bd:.2f}m.mp4'
                media.write_video(out, frames, fps=60)
                print(f'  → {out}')
            nm += 1
            ac.load_state_dict(train_w)

        # ── 7. Checkpoint every 50 rounds ────────────────────────────────────
        if rnd % 50 == 49:
            torch.save({
                'ac': ac.state_dict(), 'opt': opt.state_dict(),
                'best_w': best_w, 'best_dist': best_dist,
                'rnd': rnd, 'rl': rl, 'dl': dl, 'cll': cll, 'all_': all_,
            }, f'models/parallel_ppo_rnd{rnd}.pth')

print(f'\nDone. best_dist={best_dist:.3f}m  |  {ppo_update_count} PPO updates')

rounds:   0%|          | 1/2500 [00:01<43:44,  1.05s/it]

  rnd    0 | dist=0.318m avg10=0.318 | R=-68.3 | buf=1936 | 3015 steps/s | updates=0


rounds:   0%|          | 6/2500 [00:02<16:28,  2.52it/s]

  rnd    5 | dist=0.164m avg10=0.248 | R=-210.4 | buf=2257 | 6698 steps/s | updates=2


rounds:   0%|          | 11/2500 [00:04<16:51,  2.46it/s]

...

  rnd   40 | dist=0.184m avg10=0.174 | R=-244.1 | buf=0 | 7699 steps/s | updates=19


rounds:   2%|▏         | 46/2500 [00:19<17:48,  2.30it/s]

  rnd   45 | dist=0.372m avg10=0.218 | R=-133.2 | buf=2969 | 7681 steps/s | updates=21


rounds:   2%|▏         | 47/2500 [00:19<17:12,  2.38it/s]Traceback (most recent call last):
  File "/Users/cristianocapone/anaconda3/envs/cleanenv/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/cristianocapone/anaconda3/envs/cleanenv/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/cristianocapone/anaconda3/envs/cleanenv/lib/python3.11/concurrent/futures/process.py", line 249, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/cristianocapone/anaconda3/envs/cleanenv/lib/python3.11/multiprocessing/queues.py", line 103, in get
    res = self._recv_bytes()
          ^^^^^^^^^^^^^^^^^^
  File "/Users/cristianocapone/anaconda3/envs/cleanenv/lib/python3.11/multiprocessing/connection.py", line 216, in recv_bytes
    buf = self._recv_bytes(maxlength)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/c

KeyboardInterrupt: 

In [ ]:
# ── Results ───────────────────────────────────────────────────────────────────
w = 20
sm = lambda x: np.convolve(x, np.ones(w)/w, 'valid')

if len(dl) >= w:
    fig, ax = plt.subplots(2, 2, figsize=(12, 8))
    ax[0,0].plot(sm(dl),   color='#2ecc71'); ax[0,0].set_title('Distance (m)')
    ax[0,1].plot(sm(rl),   color='#3498db'); ax[0,1].set_title('Reward')
    ax[1,0].plot(sm(cll),  color='#e74c3c'); ax[1,0].set_title('Value Loss')
    ax[1,1].plot(sm(all_), color='#f39c12'); ax[1,1].set_title('Policy Loss')
    for a_ in ax.flat:
        a_.grid(True, alpha=0.3); a_.set_xlabel('round')
    plt.suptitle(f'Parallel PPO ({N_WORKERS} workers) | best={best_dist:.3f}m')
    plt.tight_layout()
    plt.savefig('plots/parallel_ppo_results.png', dpi=120)
    plt.show()

# ── Export best trajectory + policy header ────────────────────────────────────
if best_w:
    ac.load_state_dict(best_w)
    torch.save({"ac": best_w, "best_dist": best_dist},
               "models/parallel_ppo_best.pth")

    print("\nRecording final video + exporting trajectory...")
    bd, frames, traj, delta_traj = rollout_eval(render=True, record_traj=True)
    if frames:
        media.write_video("movies/parallel_ppo_best.mp4", frames, fps=60)
        print(f"Final: movies/parallel_ppo_best.mp4  dist={bd:.3f}m")

    # Cinematic passes for the sim2real video (1080p60)
    print("Rendering cinematic footage...")
    render_cinematic("movies/albert_night.mp4",  weights=best_w, look='night')
    render_cinematic("movies/albert_day.mp4",    weights=best_w, look='day')
    render_cinematic("movies/albert_slowmo.mp4", weights=best_w, look='night', slowmo=0.4)

    # Learning progression: same shot, checkpoints from round 0 → end
    render_progress_montage("movies/albert_learning.mp4", look='day', n_clips=8)
    export_trajectory(traj, tag="best", delta_traj=delta_traj)
    export_policy_header("models/policy.h")

    # ── One-cycle detection ───────────────────────────────────
    one_cycle_found = False
    T = traj.shape[0]
    if T > 64:
        sig   = traj[:, 0] - traj[:, 0].mean()
        fft   = np.fft.rfft(sig)
        freqs = np.fft.rfftfreq(T, d=CTRL_DT)
        dom_f = freqs[np.argmax(np.abs(fft[1:])) + 1]
        if dom_f > 0.1:
            spc = int(round(1.0 / (dom_f * CTRL_DT)))
            if T >= 2 * spc:
                export_trajectory(traj[:spc], tag="one_cycle", delta_traj=delta_traj[:spc])
                print(f"\nOne-cycle: {spc} steps ({1/dom_f:.2f}s, {dom_f:.2f} Hz)")
                one_cycle_found = True

    # ══════════════════════════════════════════════════════════
    #  ESP32 export bundle
    #  Collects policy.h + the best available trajectory.h
    #  into export/ so you can drop both files into the Arduino
    #  sketch folder and compile immediately.
    #
    #  Preference: one_cycle.h (shorter, loops cleanly on robot)
    #              → falls back to best.h if no cycle detected
    #  NB: keyed on this run's detection, not on a stale file
    #      left in trajectories/ by an earlier run.
    # ══════════════════════════════════════════════════════════
    import shutil
    os.makedirs("export", exist_ok=True)

    # policy.h — actor weights + policy_step() C function
    shutil.copy("models/policy.h", "export/policy.h")
    print("\nESP32 export bundle → export/")
    print("  Copied: models/policy.h  →  export/policy.h")

    # trajectory.h — pick one_cycle if available, otherwise best
    if one_cycle_found:
        src_traj = "trajectories/one_cycle.h"
        label    = "one_cycle"
    else:
        src_traj = "trajectories/best.h"
        label    = "best"

    shutil.copy(src_traj, "export/trajectory.h")
    print(f"  Copied: {src_traj}  →  export/trajectory.h  ({label})")

    print("\n  Sketch folder contents should be:")
    print("    Albert_AML_traj/")
    print("      Albert_AML_traj.ino")
    print("      trajectory.h          ← from export/trajectory.h")
    print("    Albert_AML_policy/")
    print("      Albert_AML_policy.ino")
    print("      policy.h              ← from export/policy.h")